# Notebook 01: Data Preprocessing & Feature Engineering

**Purpose**: Load raw CSV data, merge inventory with repair history, engineer features,
compute Asset Health Score, and create training labels for the dual ML models.

**Output**: Cleaned, preprocessed DataFrame ready for model training.

In [1]:
# === IMPORTS ===
import pandas as pd
import numpy as np
import re
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [2]:
# === CONFIGURATION ===
CURRENT_YEAR = 2026
BASE_DIR = Path.cwd().parent  # ICT_AI_DSS/
DATA_DIR = BASE_DIR / 'data'

INVENTORY_PATH = DATA_DIR / 'inv_inventory.csv'
REPAIR_PATH = DATA_DIR / 'repairhistory.csv'
DIVISION_PATH = DATA_DIR / 'division_counts.csv'

print(f'Base directory: {BASE_DIR}')
print(f'Data directory: {DATA_DIR}')

Base directory: C:\Users\User\Desktop\CaseStudy_AMSOS\ICT_AI_DSS
Data directory: C:\Users\User\Desktop\CaseStudy_AMSOS\ICT_AI_DSS\data


---
## Step 1: Load Data

In [3]:
def load_data():
    try:
        inv = pd.read_csv(INVENTORY_PATH, low_memory=False)
        repair = pd.read_csv(REPAIR_PATH, low_memory=False)
        div = pd.read_csv(DIVISION_PATH, low_memory=False)
        print(f'Inventory: {inv.shape[0]} rows, {inv.shape[1]} cols')
        print(f'Repair History: {repair.shape[0]} rows, {repair.shape[1]} cols')
        print(f'Division Counts: {div.shape[0]} rows, {div.shape[1]} cols')
        return inv, repair, div
    except Exception as e:
        print(f'Error loading data: {e}')
        raise

inv, repair, div = load_data()

Inventory: 1100 rows, 30 cols
Repair History: 56 rows, 14 cols
Division Counts: 14 rows, 2 cols


In [4]:
inv.head(3)

,id,employeeName,equipmentType,computer_specs,yearAcquired,shelfLife,brand,specifications,rangeCategory,softwareInstalled,...,actualUserSex,actualUserStatusOfEmployment,natureOfWork,remarks,office,amount,depreciation_value,mark_as_done,created_at,updated_at
0,2,Ana Baena Pedalizo,Desktop Computer,NaN,2020.0,Within 5 Year,HP,"HP 21.5"" LED monitor, Intel Core i7-8700, 8GB ...",Mid Range Level,Windows 10 Pro,...,Male,Contract of Service / Job Order,Administrative Works / Clerical,Operational,REGIONAL OFFICE,0,0,1.0,2025-10-06 05:27:20,2026-01-20 09:27:51
1,3,Ana Baena Pedalizo,Desktop Computer,NaN,2019.0,Beyond 5 year,HP,"HP, 21.5"" LED monitor, Intel Core i7-8700, 8GB...",Mid Range Level,Windows 11 Pro,...,Female,Permanent,Administrative Works / Clerical,Operational,REGIONAL OFFICE,0,0,1.0,2025-10-06 05:27:20,2026-01-20 09:27:51
2,4,Ana Baena Pedalizo,Desktop Computer,NaN,2022.0,Within 5 Year,NaN,"ESCAMING, Ryzen 3 2100MB Biostar, 2pcs 8GB RAM...",Entry/ Basic Level,Windows 10 Pro,...,Female,Contract of Service / Job Order,Technical Works,Operational,REGIONAL OFFICE,0,0,1.0,2025-10-06 05:27:20,2026-01-20 09:27:51


In [5]:
repair.head(3)

,Property No,Actual User,Equipment Type,Brand,Year Acquired,Shelf Life,Equiment ID,Total Times Repaired,SRF Track ID,Date Recorded,Time Recorded,Action Staff,Remarks,Remarks / Action Taken
0,DComputerSet-002-2024,Ana Baena A. Pedalizo,Desktop Computer,HP,2024,Within 5 Year,5,6,1009,6/23/2026,10:24:44 AM,VINCENT M. FAJARDO,Operational,Upon Checking: | | No printer software/driver...
1,DComputerSet-002-2024,Ana Baena A. Pedalizo,Desktop Computer,HP,2024,Within 5 Year,5,6,967,5/26/2026,2:43:54 PM,JOAN A. RUALES,Operational,"Upon Checking, Only Windows Screen Appears | ..."
2,DComputerSet-002-2024,Ana Baena A. Pedalizo,Desktop Computer,HP,2024,Within 5 Year,5,6,735,11/21/2025,9:38:46 AM,IVY P. CALOOY,Operational,"Upon checking, it was determined that the syst..."


In [6]:
div

,Division,Count
0,Administration Division,69.0
1,ARDMS,6.0
2,ARDTS,9.0
3,CDD,87.0
4,COA,NaN
5,ED,26.0
6,ENGP,26.0
7,Finance,26.0
8,LEGAL,13.0
9,LPDD,31.0


---
## Step 2: Merge Inventory with Repair History

Left join on `inv.id` = `repair['Equiment ID']`

In [7]:
# Normalize column types for merge
inv['id'] = pd.to_numeric(inv['id'], errors='coerce')
repair['Equiment ID'] = pd.to_numeric(repair['Equiment ID'], errors='coerce')

merged = inv.merge(
    repair,
    left_on='id',
    right_on='Equiment ID',
    how='left',
    suffixes=('', '_repair')
)

print(f'Merged shape: {merged.shape}')
print(f'Inventory with at least one repair: {merged["Equiment ID"].notna().sum()}')

Merged shape: (1121, 44)
Inventory with at least one repair: 56


---
## Step 3: Aggregate Repair History

Group by `id` to get per-asset repair statistics.

In [8]:
def aggregate_repairs(repair_df):
    agg = repair_df.groupby('Equiment ID').agg(
        total_repairs=('SRF Track ID', 'count'),
        latest_repair_date=('Date Recorded', 'max'),
        latest_remarks=('Remarks / Action Taken', lambda x: x.dropna().iloc[-1] if not x.dropna().empty else 'No remarks recorded')
    ).reset_index()

    repair_freq = repair_df.groupby('Equiment ID')['Date Recorded'].nunique().reset_index()
    repair_freq.columns = ['Equiment ID', 'repair_frequency']
    agg = agg.merge(repair_freq, on='Equiment ID', how='left')

    agg.columns = ['id', 'total_repairs', 'latest_repair_date', 'last_repair_remarks', 'repair_frequency']
    return agg

repair_agg = aggregate_repairs(repair)
print(f'Unique equipment with repairs: {len(repair_agg)}')
repair_agg.head()

Unique equipment with repairs: 35


,id,total_repairs,latest_repair_date,last_repair_remarks,repair_frequency
0,3,3,2/16/2026,Completed,1
1,5,6,8/9/2025,Upgraded operating system to windows10 and ins...,6
2,7,1,7/30/2025,Reformat Windows 10 Operating System,1
3,14,4,6/23/2026,Installed scanner driver Epson Es-580W,4
4,16,1,6/4/2025,"Computer ethernet port problem, replaced wifi ...",1


In [9]:
# Merge aggregated repairs back to inventory
inv = inv.merge(repair_agg, on='id', how='left')

# Fill NaN for assets with no repairs
inv['total_repairs'] = inv['total_repairs'].fillna(0).astype(int)
inv['repair_frequency'] = inv['repair_frequency'].fillna(0).astype(int)
inv['latest_repair_date'] = inv['latest_repair_date'].fillna('No repairs')
inv['last_repair_remarks'] = inv['last_repair_remarks'].fillna('No repairs')

print(f'Inventory after repair merge: {inv.shape}')
inv[['id', 'total_repairs', 'repair_frequency']].head()

Inventory after repair merge: (1100, 34)


,id,total_repairs,repair_frequency
0,2,0,0
1,3,3,1
2,4,0,0
3,5,6,6
4,6,0,0


---
## Step 4: Feature Engineering

### 4.1 Equipment Age & Remaining Useful Life

In [10]:
def compute_age_features(df):
    df['yearAcquired'] = pd.to_numeric(df['yearAcquired'], errors='coerce')
    df['equipment_age'] = CURRENT_YEAR - df['yearAcquired']
    df['equipment_age'] = df['equipment_age'].clip(lower=0)

    shelf_map = {
        'Within 5 Year': 5,
        'Beyond 5 year': 5,
        'within5years': 5
    }
    df['shelf_life_years'] = df['shelfLife'].map(shelf_map).fillna(5)
    df['remaining_useful_life'] = df['shelf_life_years'] - df['equipment_age']
    df['remaining_useful_life'] = df['remaining_useful_life'].clip(lower=0)
    return df

inv = compute_age_features(inv)
inv[['id', 'yearAcquired', 'equipment_age', 'shelfLife', 'remaining_useful_life']].head()

,id,yearAcquired,equipment_age,shelfLife,remaining_useful_life
0,2,2020.0,6.0,Within 5 Year,0.0
1,3,2019.0,7.0,Beyond 5 year,0.0
2,4,2022.0,4.0,Within 5 Year,1.0
3,5,2024.0,2.0,Within 5 Year,3.0
4,6,2017.0,9.0,Beyond 5 year,0.0


### 4.2 Extract CPU, RAM, Storage from Specifications

In [11]:
def extract_specs(specs):
    if pd.isna(specs) or specs is None:
        return pd.Series({'cpu': 'Unknown', 'ram_gb': 0, 'storage_gb': 0, 'has_ssd': 0, 'has_gpu': 0})
    specs = str(specs)

    # CPU extraction
    cpu_patterns = [
        r'(Intel (Core )?[iI]\d[- ][\w\d]+)',
        r'(Intel [\w-]+)',
        r'(Ryzen [\d\s\w]+)',
        r'(AMD [\w\s]+)',
        r'(Core [iI]\d)'
    ]
    cpu = 'Unknown'
    for p in cpu_patterns:
        m = re.search(p, specs)
        if m:
            cpu = m.group(1).strip()
            break

    # RAM extraction
    ram = 0
    ram_match = re.search(r'(\d+)\s*(?:GB|gb|Gb)\s*(?:RAM|ram|Ram|DDR[\d]?|memory|Memory)', specs)
    if ram_match:
        ram = int(ram_match.group(1))
    else:
        ram_match2 = re.search(r'(\d+)\s*GB', specs)
        if ram_match2:
            ram = int(ram_match2.group(1))
    # Handle cases like 8GB+8GB -> 16
    if ram <= 8:
        dual = re.findall(r'(8\s*GB(?:\+8\s*GB)?)', specs, re.IGNORECASE)
        if dual and '8GB+8GB' in specs.replace(' ', ''):
            ram = 16

    # Storage extraction
    storage = 0
    storage_match = re.search(r'(\d+)\s*(?:TB|tb)\s*(?:HDD|hdd|SATA)', specs)
    if storage_match:
        storage = int(storage_match.group(1)) * 1024
    storage_match = re.search(r'(\d+)\s*(?:GB|gb)\s*(?:HDD|hdd|SATA)', specs)
    if storage_match:
        storage += int(storage_match.group(1))
    if storage == 0:
        storage_match = re.search(r'(\d+)\s*(?:TB|tb)', specs)
        if storage_match:
            storage = int(storage_match.group(1)) * 1024
    storage_hdd_match = re.findall(r'(\d+)\s*(?:GB|gb|TB|tb)', specs)
    if storage == 0 and storage_hdd_match:
        for s in storage_hdd_match:
            val = int(s)
            if val >= 120:
                storage = val
                break

    # SSD detection
    has_ssd = 1 if re.search(r'SSD|ssd|NVMe|nvme|Nvme', specs) else 0

    # GPU detection
    has_gpu = 1 if re.search(r'(?:GeForce|GTX|RTX|Radeon|Graphics|GPU|NVIDIA|MX\d{3})', specs) else 0

    return pd.Series({'cpu': cpu, 'ram_gb': ram, 'storage_gb': storage, 'has_ssd': has_ssd, 'has_gpu': has_gpu})

specs_df = inv['specifications'].apply(extract_specs)
inv = pd.concat([inv, specs_df], axis=1)
print('Specs extraction complete')
inv[['id', 'specifications', 'cpu', 'ram_gb', 'storage_gb', 'has_ssd', 'has_gpu']].head()

Specs extraction complete


,id,specifications,cpu,ram_gb,storage_gb,has_ssd,has_gpu
0,2,"HP 21.5"" LED monitor, Intel Core i7-8700, 8GB ...",Intel Core i7-8700,8,1024,0,0
1,3,"HP, 21.5"" LED monitor, Intel Core i7-8700, 8GB...",Intel Core i7-8700,8,1024,0,0
2,4,"ESCAMING, Ryzen 3 2100MB Biostar, 2pcs 8GB RAM...",Ryzen 3 2100MB Biostar,8,500,1,0
3,5,"All in One Desktop HP, 24-cb1049d, CarmenB24,I...",Intel Core i5-1235U,8,512,1,0
4,6,"Acer Veriton, Intel Core i5-4460 8GB 1TB HDD, ...",Intel Core i5-4460,8,1024,0,0


### 4.3 Extract Windows & Office Version

In [12]:
def extract_software_versions(df):
    def get_windows_version(sw):
        if pd.isna(sw):
            return 'Unknown'
        sw = str(sw)
        m = re.search(r'Windows\s*[\d.]+\s*(?:Pro|Home|Enterprise|Single)?', sw)
        return m.group(0) if m else 'Unknown'

    def get_office_version(sw):
        if pd.isna(sw):
            return 'Unknown'
        sw = str(sw)
        m = re.search(r'MS Office\s*[\w\s]+\d{4}', sw)
        if not m:
            m = re.search(r'Office\s*(?:Pro Plus|Standard|Student|Home)?\s*\d{4}', sw)
        return m.group(0) if m else 'Unknown'

    df['windows_version'] = df['softwareInstalled'].apply(get_windows_version)
    df['office_version'] = df['softwareInstalled'].apply(get_office_version)
    return df

inv = extract_software_versions(inv)
inv[['id', 'softwareInstalled', 'windows_version', 'office_version']].head()

,id,softwareInstalled,windows_version,office_version
0,2,Windows 10 Pro,Windows 10 Pro,Unknown
1,3,Windows 11 Pro,Windows 11 Pro,Unknown
2,4,Windows 10 Pro,Windows 10 Pro,Unknown
3,5,Windows 11 Home Single,Windows 11 Home,Unknown
4,6,Windows 10 Pro,Windows 10 Pro,Unknown


### 4.4 Depreciation Percentage & License Risk

In [13]:
def compute_depreciation_and_license(df):
    # Depreciation % using straight-line over 5 years
    df['depreciation_pct'] = (df['equipment_age'] / df['shelf_life_years']) * 100
    df['depreciation_pct'] = df['depreciation_pct'].clip(upper=100)

    # License risk: Evaluation > Perpetual
    license_risk_map = {
        'Evaluation Copy': 2,
        'Evaluation Copy ': 2,
        'EVALUATION COPY': 2,
        'Perpetual Copy': 0,
        'Perpetual Copy ': 0
    }
    df['license_risk'] = df['licensingModel'].map(license_risk_map).fillna(1)
    return df

inv = compute_depreciation_and_license(inv)
inv[['id', 'equipment_age', 'shelf_life_years', 'depreciation_pct', 'licensingModel', 'license_risk']].head()

,id,equipment_age,shelf_life_years,depreciation_pct,licensingModel,license_risk
0,2,6.0,5.0,100.0,Evaluation Copy,2.0
1,3,7.0,5.0,100.0,Perpetual Copy,0.0
2,4,4.0,5.0,80.0,Evaluation Copy,2.0
3,5,2.0,5.0,40.0,Evaluation Copy,2.0
4,6,9.0,5.0,100.0,Perpetual Copy,0.0


### 4.5 Performance Score from Specs

In [14]:
def compute_performance_score(df):
    # Score based on RAM, CPU tier, SSD, GPU
    score = 0
    # RAM score (max 30)
    if df['ram_gb'] >= 16:
        score += 30
    elif df['ram_gb'] >= 8:
        score += 20
    elif df['ram_gb'] >= 4:
        score += 10

    # CPU score (max 30)
    cpu_str = str(df['cpu']).lower()
    if any(x in cpu_str for x in ['i7', 'i9', 'ryzen 7', 'ryzen 9']):
        score += 30
    elif any(x in cpu_str for x in ['i5', 'ryzen 5']):
        score += 20
    elif any(x in cpu_str for x in ['i3', 'ryzen 3']):
        score += 10
    else:
        score += 5

    # SSD bonus (max 25)
    score += 25 if df['has_ssd'] else 0

    # GPU bonus (max 15)
    score += 15 if df['has_gpu'] else 0

    return min(score, 100)

inv['performance_score'] = inv.apply(compute_performance_score, axis=1)
inv[['id', 'cpu', 'ram_gb', 'has_ssd', 'has_gpu', 'performance_score']].head()

,id,cpu,ram_gb,has_ssd,has_gpu,performance_score
0,2,Intel Core i7-8700,8,0,0,50
1,3,Intel Core i7-8700,8,0,0,50
2,4,Ryzen 3 2100MB Biostar,8,1,0,55
3,5,Intel Core i5-1235U,8,1,0,65
4,6,Intel Core i5-4460,8,0,0,40


---
## Step 5: Asset Health Score & Category

Weights: Age 30%, Repair Frequency 25%, Performance 20%, Depreciation 10%, License 10%, Remarks 5%

In [15]:
def compute_health_score(df):
    # Age score: inverse - newer is better
    max_age = df['equipment_age'].max() if df['equipment_age'].max() > 0 else 1
    age_score = ((1 - df['equipment_age'] / max_age) * 100).clip(0, 100)

    # Repair score: inverse - fewer repairs is better
    max_repairs = df['total_repairs'].max() if df['total_repairs'].max() > 0 else 1
    repair_score = ((1 - df['total_repairs'] / max_repairs) * 100).clip(0, 100)

    # Performance score already 0-100
    perf_score = df['performance_score']

    # Depreciation score: inverse - less depreciated is better
    dep_score = (100 - df['depreciation_pct']).clip(0, 100)

    # License score: inverse - lower risk is better
    license_score = ((1 - df['license_risk'] / 2) * 100).clip(0, 100)

    # Remarks score
    def remark_score(remarks):
        if pd.isna(remarks) or remarks == 'No repairs':
            return 80
        remarks = str(remarks).lower()
        if any(w in remarks for w in ['unserviceable', 'replace', 'beyond', 'failing', 'broken']):
            return 20
        if any(w in remarks for w in ['operational', 'working', 'functional']):
            return 80
        return 50

    remarks_sc = df['remarks'].apply(remark_score)

    health = (
        age_score * 0.30 +
        repair_score * 0.25 +
        perf_score * 0.20 +
        dep_score * 0.10 +
        license_score * 0.10 +
        remarks_sc * 0.05
    )

    return health.round(2).fillna(50)

inv['asset_health_score'] = compute_health_score(inv)

def health_category(score):
    if score >= 90:
        return 'Excellent'
    elif score >= 80:
        return 'Good'
    elif score >= 60:
        return 'Aging'
    elif score >= 40:
        return 'Poor'
    else:
        return 'Replace'

inv['health_category'] = inv['asset_health_score'].apply(health_category)
inv[['id', 'asset_health_score', 'health_category']].head(10)

,id,asset_health_score,health_category
0,2,58.41,Poor
1,3,54.15,Poor
2,4,64.94,Aging
3,5,49.47,Poor
4,6,61.12,Aging
5,7,46.98,Poor
6,8,59.94,Poor
7,9,56.18,Poor
8,10,40.29,Poor
9,11,56.18,Poor


---
## Step 6: Create ML Labels (Replacement Priority)

Business rules using Age, Repair Count, Health Score, Shelf Life, Depreciation, License.

In [16]:
def create_replacement_label(row):
    age = row['equipment_age']
    repairs = row['total_repairs']
    health = row['asset_health_score']
    shelf = row['remaining_useful_life']
    dep = row['depreciation_pct']
    license_risk = row['license_risk']

    # Critical: Old, many repairs, very low health
    if (age >= 6 and repairs >= 2) or (age >= 4 and health < 35) or (health < 25) or (dep >= 90):
        return 'Critical'
    # High: Beyond useful life or high depreciation
    if (shelf <= 1) or (dep >= 70 and health < 50) or (age >= 6) or (repairs >= 3):
        return 'High'
    # Medium: Aging but still functional
    if (health < 70) or (age >= 4 and dep >= 50) or (license_risk >= 2) or (repairs >= 1):
        return 'Medium'
    # Low: All others
    return 'Low'

inv['replacement_priority'] = inv.apply(create_replacement_label, axis=1)

print('Replacement Priority Distribution:')
print(inv['replacement_priority'].value_counts())

Replacement Priority Distribution:
replacement_priority
Critical    481
Low         349
Medium      148
High        122
Name: count, dtype: int64


In [17]:
# Preview labeled records
inv[['id', 'equipmentType', 'equipment_age', 'total_repairs', 'asset_health_score',
     'replacement_priority']].head(15)

,id,equipmentType,equipment_age,total_repairs,asset_health_score,replacement_priority
0,2,Desktop Computer,6.0,0,58.41,Critical
1,3,Desktop Computer,7.0,3,54.15,Critical
2,4,Desktop Computer,4.0,0,64.94,High
3,5,Desktop Computer,2.0,6,49.47,High
4,6,Desktop Computer,9.0,0,61.12,Critical
5,7,Printer,7.0,1,46.98,Critical
6,8,Desktop Computer,4.0,0,59.94,High
7,9,Scanner,5.0,0,56.18,Critical
8,10,Desktop Computer,14.0,0,40.29,Critical
9,11,Interactive Kiosk / Smart TV,5.0,0,56.18,Critical


---
## Step 7: Encode Categorical Features & Handle Missing Values

In [18]:
def prepare_ml_features(df):
    ml_df = df.copy()

    # Select and create feature columns
    features = pd.DataFrame()

    features['equipment_age'] = ml_df['equipment_age']
    features['total_repairs'] = ml_df['total_repairs']
    features['repair_frequency'] = ml_df['repair_frequency']
    features['remaining_useful_life'] = ml_df['remaining_useful_life']
    features['depreciation_pct'] = ml_df['depreciation_pct']
    features['license_risk'] = ml_df['license_risk']
    features['performance_score'] = ml_df['performance_score']
    features['ram_gb'] = ml_df['ram_gb']
    features['storage_gb'] = ml_df['storage_gb']
    features['has_ssd'] = ml_df['has_ssd']
    features['has_gpu'] = ml_df['has_gpu']

    # Encode equipmentType
    type_dummies = pd.get_dummies(ml_df['equipmentType'], prefix='type')
    features = pd.concat([features, type_dummies], axis=1)

    # Encode brand
    ml_df['brand_clean'] = ml_df['brand'].fillna('Unknown').str.strip()
    ml_df['brand_clean'] = ml_df['brand_clean'].replace(['', 'N/A', '0'], 'Unknown')
    brand_dummies = pd.get_dummies(ml_df['brand_clean'], prefix='brand')
    features = pd.concat([features, brand_dummies], axis=1)

    # Encode statusOfEmployment
    ml_df['status_clean'] = ml_df['statusOfEmployment'].fillna('Unknown').str.strip()
    status_dummies = pd.get_dummies(ml_df['status_clean'], prefix='status')
    features = pd.concat([features, status_dummies], axis=1)

    # Encode natureOfWork
    ml_df['nature_clean'] = ml_df['natureOfWork'].fillna('Unknown').str.strip()
    nature_dummies = pd.get_dummies(ml_df['nature_clean'], prefix='nature')
    features = pd.concat([features, nature_dummies], axis=1)

    # Encode officeDivision
    ml_df['div_clean'] = ml_df['officeDivision'].fillna('Unknown').str.strip()
    div_dummies = pd.get_dummies(ml_df['div_clean'], prefix='division')
    features = pd.concat([features, div_dummies], axis=1)

    # Ensure all numeric
    features = features.fillna(0)

    return features

feature_df = prepare_ml_features(inv)
print(f'Feature matrix: {feature_df.shape[0]} rows, {feature_df.shape[1]} cols')
feature_df.head(3)

Feature matrix: 1100 rows, 268 cols


,equipment_age,total_repairs,repair_frequency,remaining_useful_life,depreciation_pct,license_risk,performance_score,ram_gb,storage_gb,has_ssd,...,division_ED,division_ENGP,division_Finance,division_LEGAL,division_LPDD,division_Legal,division_ORED,division_PMD,division_RSCIG,division_SMD
0,6.0,0,0,0.0,100.0,2.0,50,8,1024,0,...,False,False,True,False,False,False,False,False,False,False
1,7.0,3,1,0.0,100.0,0.0,50,8,1024,0,...,False,False,True,False,False,False,False,False,False,False
2,4.0,0,0,1.0,80.0,2.0,55,8,500,1,...,False,False,True,False,False,False,False,False,False,False


In [19]:
# Verify no missing values
missing = feature_df.isnull().sum().sum()
print(f'Missing values in feature matrix: {missing}')

# Create target variables
target_classifier = inv['replacement_priority'].values
target_regressor = inv['asset_health_score'].values

print(f'Classifier target distribution:')
print(pd.Series(target_classifier).value_counts())
print(f'\nRegressor target stats:')
print(f'  Mean: {target_regressor.mean():.2f}')
print(f'  Std:  {target_regressor.std():.2f}')
print(f'  Min:  {target_regressor.min():.2f}')
print(f'  Max:  {target_regressor.max():.2f}')

Missing values in feature matrix: 0
Classifier target distribution:
Critical    481
Low         349
Medium      148
High        122
Name: count, dtype: int64

Regressor target stats:
  Mean: 63.05
  Std:  12.88
  Min:  29.85
  Max:  94.00


---
## Step 8: Save Preprocessed Data for Model Training

In [20]:
import joblib

OUTPUT_DIR = BASE_DIR / 'outputs'
MODEL_DIR = BASE_DIR / 'models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save feature columns for inference
feature_columns = list(feature_df.columns)
joblib.dump(feature_columns, MODEL_DIR / 'feature_columns.pkl')
print(f'Saved {len(feature_columns)} feature columns')

# Save preprocessed data for training
preprocessed = {
    'features': feature_df,
    'target_classifier': target_classifier,
    'target_regressor': target_regressor,
    'feature_columns': feature_columns
}
joblib.dump(preprocessed, MODEL_DIR / 'preprocessed_data.pkl')
print('Preprocessed data saved successfully')

Saved 268 feature columns


Preprocessed data saved successfully


In [21]:
# Also save the full inventory with all engineered features for later notebooks
inv.to_pickle(BASE_DIR / 'data' / 'inventory_engineered.pkl')
print('Engineered inventory saved')

Engineered inventory saved


---
## Summary

Notebook 01 completed:
- ✅ Loaded 3 CSV datasets
- ✅ Merged inventory with repair history
- ✅ Aggregated repair statistics per asset
- ✅ Feature engineering (age, specs, depreciation, license, performance)
- ✅ Asset Health Score (0-100) with 6 weighted factors
- ✅ Health Category (Excellent / Good / Aging / Poor / Replace)
- ✅ Replacement Priority labels (Critical / High / Medium / Low)
- ✅ Encoded categorical features → ready for ML
- ✅ Preprocessed data saved for Notebook 02